# EOQ Lab — Broader National K-12 Data Collection

**Scope:** National U.S. public education data **beyond** civil rights and ed-tech.

**Themes in this notebook:**
- Enrollment & demographics (NCES CCD)
- District finance (NCES F-33)
- Staffing & facilities (School Pulse Panel, FRSS)
- College readiness & programs (SPP, FRSS)
- CRDC topic inventory (enrollment, AP, graduation — already extracted)

**Prerequisite:** Run `collect_federal_crdc_edtech.ipynb` first (CRDC zips on disk).

Run cells **top to bottom**.


In [1]:
%pip install -q requests beautifulsoup4 pandas openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\saihaj\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


---
## Step 0 — Project paths & imports


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from federal_collect import ensure_federal_layout, extract_crdc_zips, refresh_manifest
from broad_k12_collect import (
    catalog_crdc_extracted,
    discover_frss_broad_k12,
    discover_nces_ccd_downloads,
    discover_spp_broad_k12,
    download_catalog_rows,
)

PATHS = ensure_federal_layout(PROJECT_ROOT)
DOWNLOAD_LOG = PATHS["logs_dir"] / "download_log.jsonl"
MANIFEST_CSV = PATHS["logs_dir"] / "manifest.csv"
BROAD_DISCOVERED = PATHS["logs_dir"] / "federal_broad_k12_discovered.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Federal root:", PATHS["federal_root"].resolve())
print("CRDC extracted:", PATHS["crdc_extracted_root"].resolve())
print("SPP broad:", (PATHS["spp_root"] / "broad_k12").resolve())


Project root: C:\Users\saihaj\Documents\Data Collection - EOQ Lab
Federal root: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\federal
CRDC extracted: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\federal\crdc_extracted
SPP broad: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\federal\spp\broad_k12


---
## Phase 1 — Catalog CRDC extracts (enrollment, AP, graduation, …)

Lists topic CSVs already unpacked from CRDC zips into `data/raw/federal/crdc_extracted/`.
No download — inventory only.


In [2]:
crdc_catalog = catalog_crdc_extracted(PATHS["crdc_extracted_root"])
if crdc_catalog.empty:
    print("No CRDC extracts found. Run collect_federal_crdc_edtech.ipynb Phase 6 first.")
else:
    print(f"CRDC extracted files: {len(crdc_catalog)}")
    print("\nBy category:")
    print(crdc_catalog.groupby("category").size())
    print("\nBy year:")
    print(crdc_catalog.groupby("year").size())
    display(crdc_catalog.groupby(["year", "category"]).size().unstack(fill_value=0))
    crdc_catalog.head(15)


CRDC extracted files: 93

By category:
category
discipline     14
enrollment     22
financials      1
other          33
test_scores    23
dtype: int64

By year:
year
2015-16     4
2017-18    54
2020-21    35
dtype: int64


category,discipline,enrollment,financials,other,test_scores
year,,,,,
2015-16,0,0,0,4,0
2017-18,7,19,1,15,12
2020-21,7,3,0,14,11


---
## Phase 2 — Refresh CRDC extracts (if you added new zips)

Re-unpacks any CRDC zips on disk. Skips files already extracted.


In [3]:
extract_records = extract_crdc_zips(
    PROJECT_ROOT,
    PATHS["crdc_extracted_root"],
    DOWNLOAD_LOG,
)
print(
    f"new: {sum(1 for r in extract_records if r['status']=='extracted')}, "
    f"skipped: {sum(1 for r in extract_records if r['status']=='skipped_exists')}, "
    f"failed: {sum(1 for r in extract_records if r['status']=='failed')}"
)


new: 0, skipped: 97, failed: 0


---
## Phase 3 — NCES Common Core of Data (CCD)

Discovers newest **school universe**, **state nonfiscal**, and **district finance (F-33)** zips from NCES catalog pages.


In [4]:
ccd_df = discover_nces_ccd_downloads()
print(f"NCES CCD downloads queued: {len(ccd_df)}")
ccd_df


NCES CCD downloads queued: 3


,source,category,dataset_title,format,url,description
0,NCES/CCD,enrollment,CCD Public School Universe,ZIP,https://nces.ed.gov/ccd/Data/zip/ccd_sch_029_1...,NCES CCD Public School Universe (national CSV ...
1,NCES/CCD,enrollment,CCD State Nonfiscal Survey,ZIP,https://nces.ed.gov/ccd/data/zip/st991bxls.zip,NCES CCD State Nonfiscal Survey (state enrollm...
2,NCES/CCD,financials,CCD School District Finance (F-33),ZIP,https://nces.ed.gov/ccd/data/zip/Sdf18_1a.zip,NCES CCD district finance / F-33 (national)


In [5]:
ccd_records = download_catalog_rows(
    ccd_df, PROJECT_ROOT, PATHS["federal_root"], PATHS["spp_root"], DOWNLOAD_LOG
)
print(f"downloaded: {sum(1 for r in ccd_records if r['status']=='downloaded')}, "
      f"skipped: {sum(1 for r in ccd_records if r['status']=='skipped_exists')}, "
      f"failed: {sum(1 for r in ccd_records if r['status']=='failed')}")


downloaded: 1, skipped: 2, failed: 0


---
## Phase 4 — FRSS surveys (facilities, dual credit, English learners)

Direct NCES downloads — stable URLs.


In [6]:
frss_df = discover_frss_broad_k12()
frss_records = download_catalog_rows(
    frss_df, PROJECT_ROOT, PATHS["federal_root"], PATHS["spp_root"], DOWNLOAD_LOG
)
print(f"FRSS — downloaded: {sum(1 for r in frss_records if r['status']=='downloaded')}, "
      f"skipped: {sum(1 for r in frss_records if r['status']=='skipped_exists')}")


FRSS — downloaded: 4, skipped: 0


---
## Phase 5 — School Pulse Panel (staffing, facilities, college readiness)

Topics **not** already collected in the civil-rights/ed-tech notebook.


In [7]:
spp_df = discover_spp_broad_k12()
print(f"SPP broad K-12 files queued: {len(spp_df)}")
spp_df.head(10)


SPP broad K-12 files queued: 15


,source,category,dataset_title,format,url,description,dest_filename
0,NCES/SPP,other,School Pulse Panel — AfterSchoolPrograms,XLSX,https://nces.ed.gov/surveys/spp/docs/release/A...,SPP (K-12 broad): AfterSchoolPrograms — nation...,AfterSchoolPrograms.xlsx
1,NCES/SPP,other,School Pulse Panel — Arts,XLSX,https://nces.ed.gov/surveys/spp/docs/release/A...,SPP (K-12 broad): Arts — national school surve...,Arts.xlsx
2,NCES/SPP,test_scores,School Pulse Panel — Assessment,XLSX,https://nces.ed.gov/surveys/spp/docs/release/A...,SPP (K-12 broad): Assessment — national school...,Assessment.xlsx
3,NCES/SPP,test_scores,School Pulse Panel — CCReadiness,XLSX,https://nces.ed.gov/surveys/spp/docs/release/C...,SPP (K-12 broad): CCReadiness — national schoo...,CCReadiness.xlsx
4,NCES/SPP,other,School Pulse Panel — FamilyEngagement,XLSX,https://nces.ed.gov/surveys/spp/docs/release/F...,SPP (K-12 broad): FamilyEngagement — national ...,FamilyEngagement.xlsx
5,NCES/SPP,financials,School Pulse Panel — FoodServices,XLSX,https://nces.ed.gov/surveys/spp/docs/release/F...,SPP (K-12 broad): FoodServices — national scho...,FoodServices.xlsx
6,NCES/SPP,other,School Pulse Panel — Housing,XLSX,https://nces.ed.gov/surveys/spp/docs/release/H...,SPP (K-12 broad): Housing — national school su...,Housing.xlsx
7,NCES/SPP,other,School Pulse Panel — PhysEd,XLSX,https://nces.ed.gov/surveys/spp/docs/release/P...,SPP (K-12 broad): PhysEd — national school sur...,PhysEd.xlsx
8,NCES/SPP,financials,School Pulse Panel — SchoolFac,XLSX,https://nces.ed.gov/surveys/spp/docs/release/S...,SPP (K-12 broad): SchoolFac — national school ...,SchoolFac.xlsx
9,NCES/SPP,teachers,School Pulse Panel — StaffVac,XLSX,https://nces.ed.gov/surveys/spp/docs/release/S...,SPP (K-12 broad): StaffVac — national school s...,StaffVac.xlsx


In [8]:
spp_records = download_catalog_rows(
    spp_df, PROJECT_ROOT, PATHS["federal_root"], PATHS["spp_root"], DOWNLOAD_LOG
)
print(f"SPP — downloaded: {sum(1 for r in spp_records if r['status']=='downloaded')}, "
      f"skipped: {sum(1 for r in spp_records if r['status']=='skipped_exists')}, "
      f"failed: {sum(1 for r in spp_records if r['status']=='failed')}")


SPP — downloaded: 15, skipped: 0, failed: 0


---
## Phase 6 — Refresh manifest & summary


In [9]:
if "PATHS" not in globals():
    raise NameError("Run Step 0 first.")

manifest = refresh_manifest(DOWNLOAD_LOG, MANIFEST_CSV)
fed = manifest[manifest["state"] == "federal"] if not manifest.empty else pd.DataFrame()

summary_rows = [
    {"location": "data/raw/federal/crdc_extracted/", "files": sum(1 for p in PATHS["crdc_extracted_root"].rglob("*") if p.is_file())},
    {"location": "data/raw/federal/spp/broad_k12/", "files": sum(1 for p in (PATHS["spp_root"] / "broad_k12").rglob("*") if p.is_file())},
    {"location": "data/raw/federal/enrollment/", "files": sum(1 for p in (PATHS["federal_root"] / "enrollment").rglob("*") if p.is_file())},
    {"location": "data/raw/federal/financials/", "files": sum(1 for p in (PATHS["federal_root"] / "financials").rglob("*") if p.is_file())},
    {"location": "data/raw/federal/teachers/", "files": sum(1 for p in (PATHS["federal_root"] / "teachers").rglob("*") if p.is_file())},
    {"location": "data/raw/federal/ (all)", "files": sum(1 for p in PATHS["federal_root"].rglob("*") if p.is_file())},
]

print("=== Broader K-12 collection summary ===")
print(pd.DataFrame(summary_rows).to_string(index=False))

if not fed.empty:
    print("\nFederal manifest by category:")
    print(fed.groupby("category").size())

print(f"\nManifest: {MANIFEST_CSV}")
print("Regenerate docs: python scripts/generate_docs.py")


=== Broader K-12 collection summary ===
                        location  files
data/raw/federal/crdc_extracted/     93
 data/raw/federal/spp/broad_k12/     15
    data/raw/federal/enrollment/     36
    data/raw/federal/financials/      3
      data/raw/federal/teachers/     10
         data/raw/federal/ (all)    482

Federal manifest by category:
category
discipline     168
enrollment      68
financials      10
other          434
teachers        12
test_scores    246
dtype: int64

Manifest: c:\Users\saihaj\Documents\Data Collection - EOQ Lab\logs\manifest.csv
Regenerate docs: python scripts/generate_docs.py
